[DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) (C) Fabrizio Smeraldi, 2020,2024 ([f.smeraldi@qmul.ac.uk](mailto:f.smeraldi@qmul.ac.uk) - [web](http://www.eecs.qmul.ac.uk/~fabri/)). This notebook is released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/).

# DIY Disease Tracking Dashboard

In [17]:
from IPython.display import clear_output
import ipywidgets as wdg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import json

In [18]:
%matplotlib inline
# make figures larger
plt.rcParams['figure.dpi'] = 100

In [19]:
# Create wrapper for file
class APIwrapper:
    # class variables shared among all instances
    _access_point = "https://api.ukhsa-dashboard.data.gov.uk"
    _last_access = 0.0 # time of last api access
    
    def __init__(self, theme, sub_theme, topic, geography_type, geography, metric):
        # build the path with all the required structure parameters.
        url_path = (f"/themes/{theme}/sub_themes/{sub_theme}/topics/{topic}/geography_types/" +
                  f"{geography_type}/geographies/{geography}/metrics/{metric}")
        self._start_url = APIwrapper._access_point + url_path # creates full URL
        self._filters = None
        self._page_size = -1
        self.count = None # will contain the number of items

    def get_page(self, filters= {}, page_size= 5):
        if page_size > 365: 
            raise ValueError("Max supported page size is 365") # Check page size is within range
        
        # restart from first page if page or filters have changed
        if filters != self._filters or page_size != self._page_size:
            self._filters = filters
            self._page_size = page_size
            self._next_url = self._start_url
        
        # signal the end of data condition
        if self._next_url == None: 
            return [] # we already fetched the last page
        
        # rate limiting to avoid bans
        curr_time = time.time() # Unix time: number of seconds since the last time the same data was requested
        deltat = curr_time - APIwrapper._last_access
        if deltat < 0.33: # max 3 requests/second
            time.sleep(0.33 - deltat)
        APIwrapper._last_access = curr_time #new '_last_access' is the time the most recent data request was made
       
        # build parameter dictionary by removing all the None
        parameters = {x:y for x, y in filters.items() if y != None}
        parameters['page_size'] = page_size  # values from filters and adding page_size
        
        # The .json() method decodes the response into Python object (dictionaries, lists; 'null' values are translated as None).
        response = requests.get(self._next_url, params = parameters).json() # access the API and download the next page of data. 
        self._next_url = response['next'] # update URL so we fetch the next page
        self.count = response['count'] # sets the count attribute to the total number of items available for this query. 
        return response['results'] # data is in the nested 'results' list 

    def get_all_pages(self, filters = {}, page_size = 365):
        self._filters = filters
        self._page_size = page_size
        self._next_url = self._start_url
        self.count = None
        
        data = [] # build up all data here
        while True:
            # use get_page to do the job, including the pacing
            next_page = self.get_page(filters, page_size) # does all of the above
            if next_page == []:
                break # we are done
            data.extend(next_page) # adds all the elements of next_page into the data.
        return data

In [20]:
# gets unpacked when instantiating API wrapper to provide the required arguments for __init__
structure_cases = {
    "theme": "infectious_disease", 
    "sub_theme": "vaccine_preventable",
    "topic": "Measles",
    "geography_type": "Nation", 
    "geography": "England",
    "metric": "measles_cases_casesByOnsetWeek",
}

# dict required as parameter for get_page/ get_all_pages methods
filters_cases = {
    "stratum": None, 
    "sex": None, 
    "year": 2025, # starts in 2023 for Measles
    "month": None, 
    "epiweek": None, 
    "date": None, # format YYYY-MM-DD e.g. 2020-07-20
    "in_reporting_delay_period": None,
}

In [21]:
# instantiate APIwrapper with the unpacked structure dict as parameters (for url_path)
measles_cases = APIwrapper(**structure_cases)

# create cases_2025 variable, call get_all_pages method with the filters and page size as parameters to fill with data, returns a list of dicts
cases_2025 = measles_cases.get_all_pages(filters_cases, page_size = 365)

# convert the raw data to a json file to allow for easy retrieval if the server is not available next time
'''check reason for converting to json'''
with open('cases_2025.json', "w") as f: 
            json.dump(cases_2025, f)

In [22]:
# gets unpacked when instantiating API wrapper to provide the required arguments for __init__
structure_vaccines = {
    "theme": "immunisation", 
    "sub_theme": "childhood-vaccines",
    "topic": "MMR1",
    "geography_type": "Nation", 
    "geography": "England",
    "metric": "MMR1_coverage_coverageByYear",
}

# dict required as parameter for get_page/ get_all_pages methods
filters_vaccines = {
    "stratum": None,
    "sex": None, 
    "year": None,
    "month": None, 
    "epiweek": None, 
    "date": None,
    "in_reporting_delay_period": None,
}

In [23]:
# instantiate APIwrapper with the unpacked structure dict as parameters (for url_path)
MMR1_vaccines = APIwrapper(**structure_vaccines)

# create vaccines_percent variable, call get_all_pages method with the filters and page size as parameters to fill with data, returns a list of dicts
vaccine_percent = MMR1_vaccines.get_all_pages(filters_vaccines, page_size = 365)

# convert the raw data to a json file to allow for easy retrieval if the server is not available next time
with open('vaccine_percent.json', "w") as f: 
            json.dump(vaccine_percent, f)

In [24]:
# wrangle data function
def wrangle(data, table_variables, column_name = '', filter_nans=False): # change to True if want to remove rows with NaN values
    # make dataframe
    df = pd.DataFrame(data) 
    df = df.pivot_table(**table_variables) # define the variables that we care about for the graphs we will do
    df = df.rename_axis(column_name, axis=1) # names the column row
    if filter_nans: # can result in a small df, but might be useful still
        df.dropna(inplace = True) # removes rows with NaN values
    if not df.empty:
        df.index = pd.to_datetime(df.index) # make dates human readable
        df = df.sort_index()
    return df

# load json file for cases
with open('cases_2025.json', 'r') as f:
    cases_data_json = json.load(f)

# build wrangled dataframe from cases json
table_variables_cases = {'index': 'date', 'columns': 'age', 'values': 'metric_value'}
dataframe_cases = wrangle(cases_data_json, table_variables_cases, 'age')

# load json file for vaccines
with open('vaccine_percent.json', 'r') as f:
    vaccine_data_json = json.load(f)
    
# build wrangled dataframe from vaccines json
table_variables_vaccines = {'index': 'date', 'columns': 'stratum', 'values': 'metric_value'}
dataframe_vaccines = wrangle(vaccine_data_json, table_variables_vaccines)

In [25]:
# API access code is in this function to get new data that will replace the 'canned' data loaded from the JSON file. 
# This function will be called by 'update'.

# for cases
def access_api_cases():
    try:
        api = APIwrapper(**structure_cases) # create new instance of API wrapper
        new_data = api.get_all_pages(filters_cases) # fill new variable with same data as used before
        
        with open('new_data_cases.json', "w") as f: # make a brand new json file from the updated data
            json.dump(new_data, f) 
        with open('new_data_cases.json', "r") as f: # open new json file
            new_data_json = json.load(f)
        df_new_dataframe = wrangle(new_data_json, table_variables_cases, 'age') # wrangle the data and make it into a new dataframe from json file
        return df_new_dataframe
    except Exception as e:
        print("Error fetching data")
        print(e)
        return

# for vaccines
def access_api_vaccines():
    try:
        api = APIwrapper(**structure_vaccines) 
        new_data = api.get_all_pages(filters_vaccines)
        with open('new_data_vaccines.json', "w") as f: 
            json.dump(new_data, f)
        with open('new_data_vaccines.json', "r") as f:
            new_data_json = json.load(f)
        df_new_dataframe = wrangle(new_data_json, table_variables_vaccines) 
        return df_new_dataframe
    except Exception as e:
        print("Error fetching data")
        print(e)
        return

In [26]:
# refresh cases graph function
def refresh_cases_graph(): # this force refreshes the graph by changing one of the widget values 
    current = select_age_cases_wdg.value
    opt0 = select_age_cases_wdg.value[0]
    opt1 = select_age_cases_wdg.options[1]
    
    if current == (opt0,):
        other = (opt1,)
    else:
        other = (opt0,)
    select_age_cases_wdg.value = other # changes widget value
    select_age_cases_wdg.value = current # changes it back, refreshing the graph

# refresh vaccine graph function
def refresh_vaccines_graph(): 
    current = select_age_vaccines_wdg.value
    if current == select_age_vaccines_wdg.options[0]:
        other = select_age_vaccines_wdg.options[1]
    else:
        other = select_age_vaccines_wdg.options[0]
    select_age_vaccines_wdg.value = other 
    select_age_vaccines_wdg.value = current 

In [27]:
# list of all the age bands in the dataframe
all_age_bands = dataframe_cases.columns.tolist()

# age selector widget
select_age_cases_wdg = wdg.SelectMultiple(
    options = all_age_bands,
    value = ['all'], # want it to be this by default
    rows = 8,
    description = 'Age group',
    disabled = False
)

vaccine_ages = dataframe_vaccines.columns.tolist()

# age selector widget
select_age_vaccines_wdg = wdg.Dropdown(
    options = vaccine_ages,
    value = '24m', # want it to be this by default
    description = 'Age group',
    disabled = False
    )

In [28]:
# button callback function- cases
def update_cases(button):
    df_apidata = access_api_cases() # accesses API and wrangles data
    
    global dataframe_cases # updates global variable df used for plotting
    dataframe_cases = df_apidata 
    
    refresh_cases_graph() # refreshes graph now that the dataframe being used has been replaced with the updated one
    
    apibutton_cases.icon = "check" # shows that the update function has been completed
    apibutton_cases.disabled = True 

# button callback function- vaccines
def update_vaccines(button):
    df_apidata = access_api_vaccines() # accesses API and wrangles data
    
    global dataframe_vaccines # updates global variable df used for plotting
    dataframe_vaccines = df_apidata 
    
    refresh_vaccines_graph() # refreshes graph now that the dataframe being used has been replaced with the updated one
    
    apibutton_vaccines.icon = "check" # shows that the update function has been completed
    apibutton_vaccines.disabled = True

In [29]:
# refresh button widget for cases
apibutton_cases = wdg.Button(
    description = 'Update data on cases',
    disabled = False,
    button_style = 'info',
    tooltip = "fetch latest cases data",
    icon = 'download',
    layout = wdg.Layout(width='auto', padding='0px 20px')
    )

# refresh button widget for vaccines
apibutton_vaccines = wdg.Button(
    description = 'Update data on vaccines',
    disabled = False,
    button_style = 'info',
    tooltip = "fetch latest vaccines data",
    icon = 'download',
    layout = wdg.Layout(width='auto', padding='0px 20px')
    )


apibutton_cases.on_click(update_cases) # registers button callback function with the button- when button is selected, it updates the data
apibutton_vaccines.on_click(update_vaccines) 

Graph showing measles cases per week in England in 2025 for different age groups.

You can choose one or several age bands to view at the same time, or simple view 'all' as one line!

Refresh the data by clicking the 'update data on cases button'.

In [30]:
# plot cases graph function
def plot_cases_graph(ages):
    ages = list(ages) # converting to list as was a tuple- needs to be list for else statement
    if ages == 'all':
        df_plot = dataframe_cases
    else:
        df_plot = dataframe_cases[ages]

    df_plot.index = df_plot.index.strftime('%d %b %Y') # fixes date formatting

    df_plot.plot(kind = 'bar', figsize=(16, 8), width=0.8, ax=plt.gca()) # # plot the selected age group
    
    # edit graph formatting
    plt.xlabel('Date of reported cases', fontsize = 12)
    plt.ylabel('Confirmed measles cases', fontsize = 12)
    plt.legend(ages, fontsize = 10, title = 'Age group', title_fontsize = 12, loc = 'upper right') # show the selected ages in legend
    plt.title('Measles cases per week in England, 2025 by age group', fontsize = 15, pad = 20)
    plt.grid(alpha = 0.3)
    plt.show(block=False) # has to be false to work in Viola

In [31]:
# create graph by calling the wdg.interactive_output function with the plot_cases_graph function and the selected ages as plot_cases_graph's args
# interactive output function creates live link between the selected age and the plot graph function, so if age changes, the plot_cases_graph automatically changes
cases_graph = wdg.interactive_output(plot_cases_graph, {'ages': select_age_cases_wdg}) 
# display the graph with the widgets
display(apibutton_cases, select_age_cases_wdg, cases_graph)

Button(button_style='info', description='Update data on cases', icon='download', layout=Layout(padding='0px 20…

SelectMultiple(description='Age group', index=(7,), options=('00-01', '01-04', '05-10', '11-14', '15-24', '25-…

Output()

Graph showing the percentage of children who received the MMR1 vaccine per year in England.

You can choose either to view the percentage of children who received the vaccine by age 5, or by age 2 by selecting one of these ages from the Age Group selector.

Refresh the data by clicking the 'update data on vaccines button'.

In [32]:
# plot vaccines graph function
def plot_vaccine_graph(stratum): # takes stratum as an argument
    dataframe_vaccines[stratum].plot(ax=plt.gca())
    plt.legend(title='Age group', fontsize=9, title_fontsize=11, loc='upper left')
    
    # edit graph formatting
    plt.xlabel('Year', fontsize = 12)
    plt.ylabel('Percentage', fontsize = 12)
    plt.legend(fontsize = 10, title = 'Age group', title_fontsize = 12, loc = 'upper right')
    plt.title('Percentage of vaccinated children by age 5 in the UK', fontsize = 15, pad = 20)
    plt.grid(alpha = 0.3)
    plt.show(block=False) # has to be false to work in Viola

# create graph by calling the wdg.interactive_output function with the plot_vaccine_graph function and the stratum selected as plot_vaccine_graph's argument
vaccine_graph = wdg.interactive_output(plot_vaccine_graph, {'stratum': select_age_vaccines_wdg})

# display the graph with the widgets
display(apibutton_vaccines, select_age_vaccines_wdg, vaccine_graph)

Button(button_style='info', description='Update data on vaccines', icon='download', layout=Layout(padding='0px…

Dropdown(description='Age group', options=('24m', '5y'), value='24m')

Output()

**Author and License** Remember that if you deploy your dashboard as a Binder it will be publicly accessible. Change the copyright notice and take credit for your work! Also acknowledge your sources and the conditions of the license by including this notice: "Based on UK Government [data](https://ukhsa-dashboard.data.gov.uk/) published by the [UK Health Security Agency](https://www.gov.uk/government/organisations/uk-health-security-agency) and on the [DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) by Fabrizio Smeraldi. Released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/)."